LETS DO THIS THING NOW 

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Define the path to your dataset folder
base_dir = 'C:\\Users\\gadit\\OneDrive\\Desktop\\CROP_DEKHO\\archive (1)\Rice_Leaf_AUG' # CHANGE THIS to your folder path

# Define image dimensions (smaller = faster CPU training)
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32

<>:6: SyntaxWarning: invalid escape sequence '\R'
<>:6: SyntaxWarning: invalid escape sequence '\R'
C:\Users\gadit\AppData\Local\Temp\ipykernel_9176\669864806.py:6: SyntaxWarning: invalid escape sequence '\R'
  base_dir = 'C:\\Users\\gadit\\OneDrive\\Desktop\\CROP_DEKHO\\archive (1)\Rice_Leaf_AUG' # CHANGE THIS to your folder path


In [3]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,     # Normalize pixel values [-1,1] as required for mobilenetv2
    rotation_range = 40, # Randomnly rotate up to 40 degrees
    width_shift_range = 0.2, # Randomnly shift left/right by 20%
    height_shift_range = 0.2, # Randomnly shift up/dowm by 20%
    shear_range = 0.2, # Randomnly apply shearing
    zoom_range = 0.2,  # Randomnly zoom in/out
    horizontal_flip = True, # Randomnly flip images horizontally
    fill_mode = 'nearest', # Fill in missing pixels from rotating and shifting
    validation_split = 0.2 # Reserve 20% data for validation 

)

#Applies a transformation to an image according to given parameters.

In [4]:
#Lets load the training data 

train_generated = train_datagen.flow_from_directory(
    base_dir,
    target_size = (IMG_HEIGHT,IMG_WIDTH),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    subset = 'training',
    shuffle = True,
)

# Lets load the validation data 

validation_generated = train_datagen.flow_from_directory(
    base_dir,
    target_size = (IMG_HEIGHT, IMG_WIDTH),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    subset = 'validation',
    shuffle = False 
)

Found 3066 images belonging to 6 classes.
Found 763 images belonging to 6 classes.


we take the train data and feed it into mobilenetV2 strip off the final layer and apply our custom final dense classification block having 6 classes 

In [13]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

i_model = MobileNetV2(include_top=False,
                    weights = 'imagenet',
                    input_shape = (IMG_HEIGHT,IMG_WIDTH,3))
i_model.trainable = False
model = Sequential([
    i_model,
    GlobalAveragePooling2D(),
    Dropout(0.2),
    Dense(6,activation = 'softmax')
])

model.compile(metrics = ['accuracy'],
              optimizer = Adam(learning_rate = 0.001),
              loss = 'categorical_crossentropy')
model.fit(train_generated,
          epochs = 15,
          validation_data = validation_generated,
          callbacks = tf.keras.callbacks.EarlyStopping(
              patience = 3,
              restore_best_weights = True,
              monitor = 'val_loss'
          ))
print(model.summary())
#Lets now save the model 
model.save("rice_model_detect_diseases.keras")
print("Model is saved successfully")


Epoch 1/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 30s 288ms/step - accuracy: 0.4452 - loss: 1.4676 - val_accuracy: 0.5387 - val_loss: 1.1780
Epoch 2/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 25s 259ms/step - accuracy: 0.6057 - loss: 1.0464 - val_accuracy: 0.5819 - val_loss: 1.2379
Epoch 3/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 25s 257ms/step - accuracy: 0.6491 - loss: 0.9441 - val_accuracy: 0.5596 - val_loss: 1.1838
Epoch 4/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 25s 259ms/step - accuracy: 0.6732 - loss: 0.8576 - val_accuracy: 0.6094 - val_loss: 1.0689
Epoch 5/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 25s 259ms/step - accuracy: 0.6892 - loss: 0.8381 - val_accuracy: 0.6265 - val_loss: 1.0224
Epoch 6/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 25s 258ms/step - accuracy: 0.6996 - loss: 0.8019 - val_accuracy: 0.6081 - val_loss: 1.0521
Epoch 7/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 25s 258ms/step - accuracy: 0.7143 - loss: 0.7746 - val_accuracy: 0.6606 - val_loss: 0.9679
Epoch 8/15
96/96 ━━━━━━━━━━━━━━━━━━━━ 36s 372ms/step - accuracy: 0.7211 - loss: 0.7483 - val_accu

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,281,044 (8.70 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 15,374 (60.06 KB)

None
Model is saved successfully


In [16]:
# Lets now build the remedy part of this project 

from tensorflow.keras.preprocessing import image
 #Loading the trained model 
model = tf.keras.models.load_model("rice_model_detect_diseases.keras")

CLASS_NAMES = [
    'Bacterial_Leaf_Blight',
    'Brown_Spot',
    'Healthy_Ric_Leaf',
    'Leaf_Blast',
    'Leaf_scald',
    'Sheath_Blight'
]

REMEDY_DATABASE = {
    'Bacterial_Leaf_Blight': {
        'status': 'Diseased ⚠️',
        'cause': 'Bacterium (Xanthomonas oryzae), often spread by wind and rain in warm, humid weather.',
        'immediate_action': 'Stop applying nitrogen fertilizers immediately (urea makes the disease spread faster). Drain excess standing water from the field.',
        'treatment': 'Spray Copper Oxychloride (2.5g per liter of water) mixed with Streptocycline (0.15g per liter).',
        'prevention': 'Use resistant seed varieties and avoid clipping seedling tips during transplanting.'
    },
    'Brown_Spot': {
        'status': 'Diseased ⚠️',
        'cause': 'Fungus (Bipolaris oryzae), usually triggered by nutrient-deficient soil (lack of Potassium or Silicon).',
        'immediate_action': 'Check soil moisture and apply Muriate of Potash (Potassium fertilizer) to boost plant immunity.',
        'treatment': 'Foliar spray of Mancozeb (2.0g per liter) or Propiconazole (1.0ml per liter of water) at the initial stage of infection.',
        'prevention': 'Treat seeds with Carbendazim (2g/kg of seed) before sowing next season.'
    },
    'Leaf_Blast': {
        'status': 'Diseased ⚠️',
        'cause': 'Fungus (Magnaporthe oryzae), favored by high nitrogen levels, cloudy skies, and frequent drizzle.',
        'immediate_action': 'Avoid draining the field completely (drought stress worsens Blast) and split nitrogen fertilizer applications.',
        'treatment': 'Apply Tricyclazole 75% WP (0.6g per liter of water) or Isoprothiolane (1.5ml per liter).',
        'prevention': 'Destroy diseased crop residue after harvest and use silicon-based soil amendments.'
    },
    'Healthy_Ric_Leaf': {
        'status': 'Healthy ✅',
        'cause': 'No disease detected.',
        'immediate_action': 'No intervention needed. Continue standard irrigation and monitoring.',
        'treatment': 'None required.',
        'prevention': 'Maintain balanced NPK fertilization and inspect leaves weekly during humid weather.'
    },

    'Leaf_scald': {
        'status': 'Diseased ⚠️',
        'cause': 'Fungus (Microdochium oryzae), triggered by heavy wet weather, high nitrogen, and close plant spacing.',
        'immediate_action': 'Halt heavy nitrogen top-dressing and remove weeds around the field bunds to improve airflow.',
        'treatment': 'Spray Benomyl, Carbendazim (1.0g/liter of water), or Mancozeb (2.5g/liter) on the foliage.',
        'prevention': 'Avoid dense planting and treat seeds with hot water or fungicide prior to planting.'
    },
    'Sheath_Blight': {
        'status': 'Diseased ⚠️',
        'cause': 'Soil-borne fungus (Rhizoctonia solani), thriving in high humidity, warm temperatures (28–32°C), and dense canopies.',
        'immediate_action': 'Drain the field for a few days to reduce humidity around the lower plant stems and sheaths.',
        'treatment': 'Spray Hexaconazole 5% EC (2.0ml/liter of water) or Validamycin 3% L (2.0ml/liter) directed at the base of the plants.',
        'prevention': 'Practice alternate wetting and drying (AWD) irrigation and apply Trichoderma viride to the soil before planting.'
}
}

In [ ]:
import numpy as np

def crop_see(img_path):

    img = image.load_img(img_path, target_size=(128,128))
    img_array = image.img_to_array(img)
    img_batch = np.expand_dims(img_array,axis = 0)
    processed_img = preprocess_input(img_batch)
    predictions = model.predict(processed_img, verbose = 0)
    predicted_index = np.argmax(predictions[0])
    confidence = float(np.max(predictions[0]*100))
    predicted_class = CLASS_NAMES[predicted_index]

    if confidence < 60.0:
        print("UPLOAD A CLEARER IMAGE ")
        return
    remedy_info = REMEDY_DATABASE.get(predicted_class)
    print("=" * 50)
    print(f"🌾 CROP DIAGNOSIS REPORT")
    print("=" * 50)
    print(f"Detected Condition : {predicted_class.replace('_', ' ')}")
    print(f"AI Confidence      : {confidence:.2f}%")
    print(f"Crop Status        : {remedy_info['status']}")
    print("-" * 50)
    print(f"🔍 Cause           : {remedy_info['cause']}")
    print(f"⚡ Immediate Action: {remedy_info['immediate_action']}")
    print(f"💊 Treatment       : {remedy_info['treatment']}")
    print(f"🛡️ Prevention      : {remedy_info['prevention']}")
    print("=" * 50)

# Test it on a single image!
test_image = r"C:\Users\gadit\OneDrive\Desktop\CROP_DEKHO\test_image7.png"
crop_see(test_image)


    

🌾 CROP DIAGNOSIS REPORT
Detected Condition : Sheath Blight
AI Confidence      : 90.68%
Crop Status        : Diseased ⚠️
--------------------------------------------------
🔍 Cause           : Soil-borne fungus (Rhizoctonia solani), thriving in high humidity, warm temperatures (28–32°C), and dense canopies.
⚡ Immediate Action: Drain the field for a few days to reduce humidity around the lower plant stems and sheaths.
💊 Treatment       : Spray Hexaconazole 5% EC (2.0ml/liter of water) or Validamycin 3% L (2.0ml/liter) directed at the base of the plants.
🛡️ Prevention      : Practice alternate wetting and drying (AWD) irrigation and apply Trichoderma viride to the soil before planting.
